### Capturing the apron allocation data from sportrac and independently verified sources of teams that were "Hard-capped" at the apron thresholds as a result of thresholds.  Useful for measuring apron allocations against the first and second apron thresholds by franchise-season grain asa way to measure competitiveness and risk by GMs looking to compete in the NBA... S19 data pipeline

In [1]:

from pathlib import Path
import pandas as pd

# Define the raw data directory relative to the notebooks folder.
RAW_DIR = Path("../data/raw/Apron Allocations")

# Load the three CSVs without changing the original files.
apron_raw = pd.read_csv(
    RAW_DIR / "apron_allocation_history.csv"
)

hard_cap_raw = pd.read_csv(
    RAW_DIR / "s19_hard_cap_full_research.csv"
)

hard_cap_evidence = pd.read_csv(
    RAW_DIR / "s19_hard_cap_evidence_expanded.csv"
)

# Inspect each DataFrame's dimensions and column names.
for name, df in {
    "Spotrac allocations": apron_raw,
    "Hard-cap research": hard_cap_raw,
    "Hard-cap evidence": hard_cap_evidence
}.items():
    print(f"\n--- {name} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))



--- Spotrac allocations ---
Shape: (480, 10)
Columns: ['Rank', 'Team', 'Record', 'Total ApronAllocations', '1st ApronIs Hard-Capped?', '1st ApronSpace', '2nd ApronIs Hard-Capped?', '2nd ApronSpace', 'Year', 'Season']
Rank Team Record Total ApronAllocations 1st ApronIs Hard-Capped? 1st ApronSpace 2nd ApronIs Hard-Capped? 2nd ApronSpace Year    Season
 NaN  NaN    NaN                    NaN                      NaN            NaN                      NaN            NaN  NaN       NaN
   1  CLE  52-30           $212,404,326                      NaN   -$16,459,326                      NaN    -$4,580,326 2026 2025-2026
   2  NYK  53-29           $207,454,264                      NaN   -$11,509,264                      NaN       $369,736 2026 2025-2026

--- Hard-cap research ---
Shape: (450, 8)
Columns: ['year', 'season', 'team', 'first_apron_hard_capped', 'second_apron_hard_capped', 'verification_status', 'source_url', 'source_coverage']
 year    season team first_apron_hard_capped second_

In [2]:

# Count rows that are completely empty.
blank_rows = apron_raw.isna().all(axis=1)

print("Total raw rows:", len(apron_raw))
print("Completely blank rows:", blank_rows.sum())
print("Nonblank rows:", (~blank_rows).sum())

# Show the number of nonblank rows for each season.
# dropna=False includes missing season values in the count.
print("\nRows by season:")
print(apron_raw["Season"].value_counts(dropna=False).sort_index())

# Check how the Year and Season columns correspond.
print("\nYear/Season combinations:")
print(
    apron_raw.loc[~blank_rows, ["Year", "Season"]]
    .drop_duplicates()
    .sort_values("Season")
    .to_string(index=False)
)

# Identify partially populated rows that may be copied headers
# or other artifacts rather than actual team observations.
partial_rows = apron_raw[
    ~blank_rows &
    (
        apron_raw["Team"].isna() |
        apron_raw["Season"].isna()
    )
]

print("\nPotential malformed rows:", len(partial_rows))
print(partial_rows.head(20).to_string(index=False))


Total raw rows: 480
Completely blank rows: 16
Nonblank rows: 464

Rows by season:
Season
2011-2012    30
2012-2013    30
2013-2014    30
2014-2015    30
2015-2016    30
2016-2017    30
2017-2018    30
2018-2019    30
2019-2020    30
2020-2021    30
2021-2022    30
2022-2023    30
2023-2024    30
2024-2025    30
2025-2026    30
Season       14
NaN          16
Name: count, dtype: int64

Year/Season combinations:
Year    Season
2012 2011-2012
2013 2012-2013
2014 2013-2014
2015 2014-2015
2016 2015-2016
2017 2016-2017
2018 2017-2018
2019 2018-2019
2020 2019-2020
2021 2020-2021
2022 2021-2022
2023 2022-2023
2024 2023-2024
2025 2024-2025
2026 2025-2026
Year    Season

Potential malformed rows: 0
Empty DataFrame
Columns: [Rank, Team, Record, Total ApronAllocations, 1st ApronIs Hard-Capped?, 1st ApronSpace, 2nd ApronIs Hard-Capped?, 2nd ApronSpace, Year, Season]
Index: []


In [3]:

# Start with a copy so the original raw DataFrame remains unchanged.
apron_stage = apron_raw.copy()

# Remove completely blank separator rows.
# The Team column is populated for every legitimate franchise observation.
apron_stage = apron_stage[
    apron_stage["Team"].notna()
].copy()

# Remove repeated header rows from the concatenated tables.
# These rows contain the literal string "Team" rather than a franchise code.
apron_stage = apron_stage[
    apron_stage["Team"].ne("Team")
].copy()

# Remove columns we don't need for the analytical dataset.
# Record and Rank are not reliable or necessary S19 financial features.
apron_stage = apron_stage.drop(
    columns=[
        "Rank",
        "Record",
        "1st ApronIs Hard-Capped?",
        "2nd ApronIs Hard-Capped?"
    ]
)

# Reset the index so the cleaned rows are numbered consecutively.
apron_stage = apron_stage.reset_index(drop=True)

print("Cleaned shape:", apron_stage.shape)
print(apron_stage.head(3).to_string(index=False))


Cleaned shape: (450, 6)
Team Total ApronAllocations 1st ApronSpace 2nd ApronSpace Year    Season
 CLE           $212,404,326   -$16,459,326    -$4,580,326 2026 2025-2026
 NYK           $207,454,264   -$11,509,264       $369,736 2026 2025-2026
 GSW           $205,577,466    -$9,632,466     $2,246,534 2026 2025-2026


In [4]:

# Standardize the names to match our canonical naming conventions.
apron_stage = apron_stage.rename(columns={
    "Team": "team",
    "Total ApronAllocations": "total_apron_allocations",
    "1st ApronSpace": "first_apron_space_spotrac",
    "2nd ApronSpace": "second_apron_space_spotrac",
    "Year": "year",
    "Season": "season"
})

# Convert currency strings such as "$212,404,326" and "-$16,459,326"
# into numeric values. Removing everything except digits, decimal points,
# and minus signs preserves negative values.
money_cols = [
    "total_apron_allocations",
    "first_apron_space_spotrac",
    "second_apron_space_spotrac"
]

for col in money_cols:
    apron_stage[col] = pd.to_numeric(
        apron_stage[col].str.replace(
            r"[^0-9.\-]", "", regex=True
        ),
        errors="coerce"
    )

# Spotrac's Year represents the ending year.
# Convert it to the starting year used by our research dataset.
apron_stage["year"] = pd.to_numeric(
    apron_stage["year"]
) - 1

# Verify the resulting schema and sample values.
print(apron_stage.dtypes)
print(apron_stage.head(3).to_string(index=False))


team                            str
total_apron_allocations       int64
first_apron_space_spotrac     int64
second_apron_space_spotrac    int64
year                          int64
season                          str
dtype: object
team  total_apron_allocations  first_apron_space_spotrac  second_apron_space_spotrac  year    season
 CLE                212404326                  -16459326                    -4580326  2025 2025-2026
 NYK                207454264                  -11509264                      369736  2025 2025-2026
 GSW                205577466                   -9632466                     2246534  2025 2025-2026


In [5]:

# Each season should have exactly 30 franchise observations.
print("Rows per season:")
print(apron_stage.groupby("season").size())

# There should be no duplicate franchise-season keys.
print(
    "\nDuplicate team-season rows:",
    apron_stage.duplicated(["year", "team"]).sum()
)

# Check for missing financial values after currency conversion.
print("\nMissing values:")
print(apron_stage.isna().sum())

# Check that the derived starting year matches the season string.
season_start = apron_stage["season"].str[:4].astype(int)

print(
    "\nYear/season mismatches:",
    (apron_stage["year"] != season_start).sum()
)

# These are the minimum structural requirements.
assert len(apron_stage) == 450
assert apron_stage.duplicated(["year", "team"]).sum() == 0
assert (apron_stage["year"] == season_start).all()
assert apron_stage.groupby("season").size().eq(30).all()


Rows per season:
season
2011-2012    30
2012-2013    30
2013-2014    30
2014-2015    30
2015-2016    30
2016-2017    30
2017-2018    30
2018-2019    30
2019-2020    30
2020-2021    30
2021-2022    30
2022-2023    30
2023-2024    30
2024-2025    30
2025-2026    30
dtype: int64

Duplicate team-season rows: 0

Missing values:
team                          0
total_apron_allocations       0
first_apron_space_spotrac     0
second_apron_space_spotrac    0
year                          0
season                        0
dtype: int64

Year/season mismatches: 0


In [6]:

# Keep a separate working copy of the independently researched data.
hard_cap_stage = hard_cap_raw.copy()

# Standardize the team and season identifiers.
hard_cap_stage["team"] = hard_cap_stage["team"].str.strip().str.upper()
hard_cap_stage["season"] = hard_cap_stage["season"].str.strip()

# Validate the expected grain: one row per team per season.
print("Research rows:", len(hard_cap_stage))
print(
    "Duplicate research keys:",
    hard_cap_stage.duplicated(["year", "team"]).sum()
)

# Check whether every financial team-season key appears in the research file.
key_check = apron_stage[["year", "team"]].merge(
    hard_cap_stage[["year", "team"]],
    on=["year", "team"],
    how="outer",
    indicator=True
)

print("\nKey reconciliation:")
print(key_check["_merge"].value_counts())

# Check how much of the research is verified versus unknown.
print("\nVerification status:")
print(hard_cap_stage["verification_status"].value_counts(dropna=False))

print("\nFirst-apron status:")
print(hard_cap_stage["first_apron_hard_capped"].value_counts(dropna=False))

print("\nSecond-apron status:")
print(hard_cap_stage["second_apron_hard_capped"].value_counts(dropna=False))


Research rows: 450
Duplicate research keys: 0

Key reconciliation:
_merge
both          447
left_only       3
right_only      3
Name: count, dtype: int64

Verification status:
verification_status
unverified                     262
confirmed_source_snapshot      171
source_reported_no_hard_cap     17
Name: count, dtype: int64

First-apron status:
first_apron_hard_capped
NaN      262
True     155
False     33
Name: count, dtype: int64

Second-apron status:
second_apron_hard_capped
NaN      360
False     74
True      16
Name: count, dtype: int64


In [7]:

# Display the exact year/team combinations that failed to match.
unmatched = key_check[
    key_check["_merge"] != "both"
].copy()

print(
    unmatched.sort_values(["year", "_merge"])
    .to_string(index=False)
)


 year team     _merge
 2011  NJN  left_only
 2011  NOH  left_only
 2011  BKN right_only
 2011  NOP right_only
 2012  NOH  left_only
 2012  NOP right_only


In [8]:

# Map historical franchise abbreviations to the modern identifiers
# used by the independent hard-cap research dataset.
team_aliases = {
    "NJN": "BKN",  # New Jersey Nets -> Brooklyn Nets
    "NOH": "NOP"   # New Orleans Hornets -> New Orleans Pelicans
}

# Preserve the original abbreviation for source traceability.
apron_stage["team_spotrac"] = apron_stage["team"]

# Replace only abbreviations that appear in the mapping.
# Unlisted abbreviations remain unchanged.
apron_stage["team"] = apron_stage["team"].replace(team_aliases)

# Recheck the keys after normalization.
key_check = apron_stage[["year", "team"]].merge(
    hard_cap_stage[["year", "team"]],
    on=["year", "team"],
    how="outer",
    indicator=True
)

print(key_check["_merge"].value_counts())

# All 450 franchise-season identifiers should now match.
assert key_check["_merge"].eq("both").all()


_merge
both          450
left_only       0
right_only      0
Name: count, dtype: int64


In [9]:

# Combine Spotrac financial data with independently researched
# hard-cap classifications using the standardized team-season keys.
apron_merged = apron_stage.merge(
    hard_cap_stage,
    on=["year", "season", "team"],
    how="left",
    validate="one_to_one",
    indicator=True
)

# Confirm every Spotrac row matched exactly one research row.
print("Merged shape:", apron_merged.shape)
print(apron_merged["_merge"].value_counts())

assert len(apron_merged) == 450
assert apron_merged["_merge"].eq("both").all()

# The merge indicator is only needed for validation.
apron_merged = apron_merged.drop(columns="_merge")


Merged shape: (450, 13)
_merge
both          450
left_only       0
right_only      0
Name: count, dtype: int64


In [10]:

# Work from the merged data without changing the original DataFrame.
apron_validation = apron_merged.copy()

# Reconstruct the threshold implied by each team's allocation and space.
# Negative space means the team is above the reported threshold.
apron_validation["first_apron_implied_threshold"] = (
    apron_validation["total_apron_allocations"]
    + apron_validation["first_apron_space_spotrac"]
)

apron_validation["second_apron_implied_threshold"] = (
    apron_validation["total_apron_allocations"]
    + apron_validation["second_apron_space_spotrac"]
)

# Check whether all 30 teams in each season imply the same threshold.
threshold_audit = (
    apron_validation
    .groupby(["year", "season"])
    .agg(
        teams=("team", "nunique"),
        first_min=("first_apron_implied_threshold", "min"),
        first_max=("first_apron_implied_threshold", "max"),
        second_min=("second_apron_implied_threshold", "min"),
        second_max=("second_apron_implied_threshold", "max")
    )
    .reset_index()
)

# A zero spread means every team implies the same threshold.
threshold_audit["first_spread"] = (
    threshold_audit["first_max"] - threshold_audit["first_min"]
)

threshold_audit["second_spread"] = (
    threshold_audit["second_max"] - threshold_audit["second_min"]
)

print(threshold_audit.to_string(index=False))


 year    season  teams  first_min  first_max  second_min  second_max  first_spread  second_spread
 2011 2011-2012     30   74307000   74307000           0           0             0              0
 2012 2012-2013     30   74307000   74307000           0           0             0              0
 2013 2013-2014     30   75748000   75748000           0           0             0              0
 2014 2014-2015     30   80829000   80829000           0           0             0              0
 2015 2015-2016     30   88740000   88740000           0           0             0              0
 2016 2016-2017     30  117287000  117287000           0           0             0              0
 2017 2017-2018     30  125266000  125266000           0           0             0              0
 2018 2018-2019     30  129817000  129817000           0           0             0              0
 2019 2019-2020     30  138928000  138928000           0           0             0              0
 2020 2020-2021     

In [11]:

# Start from the already validated merged dataset.
apron_final_stage = apron_validation.copy()

# Use the verified internal arithmetic to reconstruct the
# historical first-apron threshold for each franchise-season.
apron_final_stage["first_apron_threshold"] = (
    apron_final_stage["first_apron_implied_threshold"]
)

# The second apron began in 2023-24.
# Earlier values are not applicable, rather than zero.
apron_final_stage["second_apron_threshold"] = (
    apron_final_stage["second_apron_implied_threshold"]
    .where(apron_final_stage["year"] >= 2023)
)

# Preserve the raw Spotrac second-apron space column, but create
# an analytically valid version that excludes pre-2023 placeholders.
apron_final_stage["second_apron_space"] = (
    apron_final_stage["second_apron_space_spotrac"]
    .where(apron_final_stage["year"] >= 2023)
)

# First-apron space is meaningful across the full S19 period,
# although the pre-2023 threshold had different rule terminology.
apron_final_stage["first_apron_space"] = (
    apron_final_stage["first_apron_space_spotrac"]
)

# Create financial-position indicators, distinct from hard-cap triggers.
# These identify whether allocations exceed the relevant threshold.
apron_final_stage["above_first_apron"] = (
    apron_final_stage["first_apron_space"] < 0
)

# Nullable Boolean: pre-2023 is N/A, not False.
apron_final_stage["above_second_apron"] = (
    apron_final_stage["second_apron_space"] < 0
).astype("boolean")

apron_final_stage.loc[
    apron_final_stage["year"] < 2023,
    "above_second_apron"
] = pd.NA

# Inspect the resulting financial fields.
print(
    apron_final_stage[
        [
            "season", "team",
            "total_apron_allocations",
            "first_apron_threshold",
            "first_apron_space",
            "second_apron_threshold",
            "second_apron_space",
            "above_first_apron",
            "above_second_apron"
        ]
    ].tail(10).to_string(index=False)
)


   season team  total_apron_allocations  first_apron_threshold  first_apron_space  second_apron_threshold  second_apron_space  above_first_apron  above_second_apron
2011-2012  UTA                 58183391               74307000           16123609                     NaN                 NaN              False                <NA>
2011-2012  DEN                 57150022               74307000           17156978                     NaN                 NaN              False                <NA>
2011-2012  GSW                 56742440               74307000           17564560                     NaN                 NaN              False                <NA>
2011-2012  MIN                 55406532               74307000           18900468                     NaN                 NaN              False                <NA>
2011-2012  TOR                 53148082               74307000           21158918                     NaN                 NaN              False                <NA>
2011-2012 

In [12]:

# Count confirmed positive, negative, and unknown classifications
# separately for each season.
hard_cap_coverage = (
    apron_final_stage
    .groupby("season")
    .agg(
        first_hard_capped=("first_apron_hard_capped",
                           lambda s: s.eq(True).sum()),
        first_not_hard_capped=("first_apron_hard_capped",
                               lambda s: s.eq(False).sum()),
        first_unknown=("first_apron_hard_capped",
                        lambda s: s.isna().sum()),
        second_hard_capped=("second_apron_hard_capped",
                            lambda s: s.eq(True).sum()),
        second_not_hard_capped=("second_apron_hard_capped",
                                lambda s: s.eq(False).sum()),
        second_unknown=("second_apron_hard_capped",
                        lambda s: s.isna().sum())
    )
)

print(hard_cap_coverage.to_string())


           first_hard_capped  first_not_hard_capped  first_unknown  second_hard_capped  second_not_hard_capped  second_unknown
season                                                                                                                        
2011-2012                  0                      0             30                   0                       0              30
2012-2013                  0                      0             30                   0                       0              30
2013-2014                 10                      0             20                   0                       0              30
2014-2015                 10                      0             20                   0                       0              30
2015-2016                  7                      0             23                   0                       0              30
2016-2017                  0                      0             30                   0                       0 

In [13]:

# Create a new working copy to preserve our previous validation results.
s19_stage = apron_final_stage.copy()

# The modern second-apron regime begins in the 2023-24 season.
s19_stage["second_apron_applicable"] = (
    s19_stage["year"] >= 2023
)

# Convert researched classifications to pandas nullable Boolean dtype.
# This supports True, False, and pd.NA without mixing strings and booleans.
hard_cap_cols = [
    "first_apron_hard_capped",
    "second_apron_hard_capped"
]

for col in hard_cap_cols:
    s19_stage[col] = s19_stage[col].astype("boolean")

# Explicitly mark historical second-apron classifications as N/A.
# Applicability is represented by the separate indicator above.
pre_2023 = ~s19_stage["second_apron_applicable"]

s19_stage.loc[
    pre_2023, "second_apron_hard_capped"
] = pd.NA

# Confirm no pre-2023 second-apron classifications remain.
assert s19_stage.loc[
    pre_2023, "second_apron_hard_capped"
].isna().all()

print(s19_stage[[
    "season",
    "team",
    "first_apron_hard_capped",
    "second_apron_hard_capped",
    "second_apron_applicable",
    "verification_status"
]].head(10).to_string(index=False))


   season team  first_apron_hard_capped  second_apron_hard_capped  second_apron_applicable         verification_status
2025-2026  CLE                    False                     False                     True source_reported_no_hard_cap
2025-2026  NYK                    False                      True                     True   confirmed_source_snapshot
2025-2026  GSW                    False                      True                     True   confirmed_source_snapshot
2025-2026  MIN                    False                      True                     True   confirmed_source_snapshot
2025-2026  LAL                     True                     False                     True   confirmed_source_snapshot
2025-2026  HOU                     True                     False                     True   confirmed_source_snapshot
2025-2026  NOP                     True                     False                     True   confirmed_source_snapshot
2025-2026  PHI                    False         

In [14]:

# Verify the final observation count and team-season grain.
assert len(s19_stage) == 450
assert not s19_stage.duplicated(["year", "team"]).any()

# Confirm complete financial coverage.
financial_cols = [
    "total_apron_allocations",
    "first_apron_threshold",
    "first_apron_space"
]

assert s19_stage[financial_cols].notna().all().all()

# Confirm second-apron metrics exist only when applicable.
assert s19_stage.loc[
    pre_2023, "second_apron_threshold"
].isna().all()

assert s19_stage.loc[
    pre_2023, "second_apron_space"
].isna().all()

# Validate threshold arithmetic.
assert (
    s19_stage["total_apron_allocations"]
    + s19_stage["first_apron_space"]
    == s19_stage["first_apron_threshold"]
).all()

# Check researched classification coverage by season.
coverage_summary = (
    s19_stage.groupby("season")
    .agg(
        teams=("team", "nunique"),
        first_known=(
            "first_apron_hard_capped",
            "count"
        ),
        second_known=(
            "second_apron_hard_capped",
            "count"
        )
    )
)

print(coverage_summary.to_string())
print("\nAll structural and applicability checks passed.")


           teams  first_known  second_known
season                                     
2011-2012     30            0             0
2012-2013     30            0             0
2013-2014     30           10             0
2014-2015     30           10             0
2015-2016     30            7             0
2016-2017     30            0             0
2017-2018     30            7             0
2018-2019     30            9             0
2019-2020     30           15             0
2020-2021     30           18             0
2021-2022     30           11             0
2022-2023     30           11             0
2023-2024     30           30            30
2024-2025     30           30            30
2025-2026     30           30            30

All structural and applicability checks passed.


In [16]:

from pathlib import Path

# Create the staging directory if it doesn't already exist.
STAGING_DIR = Path("../data/staging/Apron Allocation")
STAGING_DIR.mkdir(parents=True, exist_ok=True)

# Choose the columns we want to preserve in the staged dataset.
# Keep both original Spotrac distances and normalized analytical distances.
staging_columns = [
    "year",
    "season",
    "team",
    "team_spotrac",
    "total_apron_allocations",
    "first_apron_threshold",
    "first_apron_space",
    "first_apron_space_spotrac",
    "second_apron_applicable",
    "second_apron_threshold",
    "second_apron_space",
    "second_apron_space_spotrac",
    "above_first_apron",
    "above_second_apron",
    "first_apron_hard_capped",
    "second_apron_hard_capped",
    "verification_status",
    "source_url",
    "source_coverage"
]

# Select and sort the columns into a consistent franchise-season order.
s19_export = (
    s19_stage[staging_columns]
    .sort_values(["year", "team"])
    .reset_index(drop=True)
)

# Save the cleaned and reconciled dataset.
output_path = STAGING_DIR / "s19_apron_allocations_staged.csv"

s19_export.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", s19_export.shape)


Saved: ../data/staging/Apron Allocation/s19_apron_allocations_staged.csv
Shape: (450, 19)


In [17]:

# Read the saved CSV back from disk to confirm it was written correctly.
s19_reloaded = pd.read_csv(output_path)

# Check that no observations or columns were lost during export.
assert s19_reloaded.shape == (450, 19)
assert not s19_reloaded.duplicated(["year", "team"]).any()

# Verify that financial values survived the export.
assert (
    s19_reloaded["total_apron_allocations"]
    .equals(s19_export["total_apron_allocations"])
)

print("Saved-file validation passed.")
print("Shape:", s19_reloaded.shape)
print("Seasons:", s19_reloaded["season"].nunique())


Saved-file validation passed.
Shape: (450, 19)
Seasons: 15


In [18]:

# Load the existing canonical tables.
# These are our reference tables for franchise identity and team-seasons.
CANONICAL_DIR = Path("../data/canonical")

franchises = pd.read_csv(
    CANONICAL_DIR / "franchises.csv"
)

franchise_aliases = pd.read_csv(
    CANONICAL_DIR / "franchise_season_aliases.csv"
)

team_seasons = pd.read_csv(
    CANONICAL_DIR / "team_seasons.csv"
)

# Inspect each table's dimensions, column names, and sample records.
# We need the actual schemas before deciding which keys to join on.
for name, df in {
    "C02 franchises": franchises,
    "Franchise-season aliases": franchise_aliases,
    "C04 team seasons": team_seasons
}.items():
    print(f"\n--- {name} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))



--- C02 franchises ---
Shape: (30, 3)
Columns: ['franchise_id', 'current_team_name', 'current_abbreviation']
 franchise_id   current_team_name current_abbreviation
   1610612737       Atlanta Hawks                  ATL
   1610612738      Boston Celtics                  BOS
   1610612739 Cleveland Cavaliers                  CLE

--- Franchise-season aliases ---
Shape: (776, 5)
Columns: ['franchise_id', 'season', 'team_abbreviation', 'full_team_name', 'short_team_name']
 franchise_id    season team_abbreviation      full_team_name short_team_name
   1610612737 2000-2001               ATL       Atlanta Hawks           Hawks
   1610612738 2000-2001               BOS      Boston Celtics         Celtics
   1610612739 2000-2001               CLE Cleveland Cavaliers       Cavaliers

--- C04 team seasons ---
Shape: (776, 53)
Columns: ['team_name', 'wins', 'losses', 'win_pct', 'games_behind', 'points_per_game', 'opponent_points_per_game', 'simple_rating_system', 'conference', 'season', 'franchi

In [19]:

# Read the saved staging output from its actual project directory.
S19_STAGE_PATH = (
    Path("../data/staging")
    / "Apron Allocation"
    / "s19_apron_allocations_staged.csv"
)

s19_saved = pd.read_csv(S19_STAGE_PATH)

print("S19 staged shape:", s19_saved.shape)
print("S19 columns:", s19_saved.columns.tolist())

assert len(s19_saved) == 450


S19 staged shape: (450, 19)
S19 columns: ['year', 'season', 'team', 'team_spotrac', 'total_apron_allocations', 'first_apron_threshold', 'first_apron_space', 'first_apron_space_spotrac', 'second_apron_applicable', 'second_apron_threshold', 'second_apron_space', 'second_apron_space_spotrac', 'above_first_apron', 'above_second_apron', 'first_apron_hard_capped', 'second_apron_hard_capped', 'verification_status', 'source_url', 'source_coverage']


In [20]:

# Each modern abbreviation should identify one franchise.
assert franchises["current_abbreviation"].is_unique

# Each franchise should have one permanent identifier.
assert franchises["franchise_id"].is_unique

# Inspect whether any S19 team codes are missing from C02.
unmapped_codes = (
    set(s19_saved["team"])
    - set(franchises["current_abbreviation"])
)

print("Unmapped team codes:", unmapped_codes)


Unmapped team codes: set()


In [21]:

# Select only the columns needed for the identifier lookup.
# This avoids bringing unnecessary franchise metadata into C10.
franchise_lookup = franchises[
    ["franchise_id", "current_abbreviation"]
].copy()

# Rename the lookup column to match the S19 team column.
# Both sides of the merge will now use "team".
franchise_lookup = franchise_lookup.rename(
    columns={"current_abbreviation": "team"}
)

# Attach the canonical franchise_id to each S19 observation.
# validate="many_to_one" ensures each S19 row matches at most
# one franchise in the lookup table.
c10_working = s19_saved.merge(
    franchise_lookup,
    on="team",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Inspect how many rows successfully matched.
print(c10_working["_merge"].value_counts())

# Confirm every S19 observation received a franchise identifier.
assert len(c10_working) == 450
assert c10_working["_merge"].eq("both").all()
assert c10_working["franchise_id"].notna().all()

# Remove the temporary merge diagnostic column.
c10_working = c10_working.drop(columns="_merge")


_merge
both          450
left_only       0
right_only      0
Name: count, dtype: int64


In [22]:

# Extract only C04's canonical identifiers.
# drop_duplicates() produces the unique franchise-season reference keys.
c04_keys = team_seasons[
    ["franchise_id", "season"]
].drop_duplicates()

# Check C04's expected key uniqueness.
assert len(c04_keys) == len(team_seasons)

# Check that C10 also has exactly one row per franchise-season.
assert not c10_working.duplicated(
    ["franchise_id", "season"]
).any()

# Compare all 450 S19 keys with C04.
key_audit = c10_working[
    ["franchise_id", "season"]
].merge(
    c04_keys,
    on=["franchise_id", "season"],
    how="left",
    validate="one_to_one",
    indicator=True
)

print(key_audit["_merge"].value_counts())

# Every C10 key must exist in C04.
assert key_audit["_merge"].eq("both").all()

print("\nC10 franchise-season mapping passed.")


_merge
both          450
left_only       0
right_only      0
Name: count, dtype: int64

C10 franchise-season mapping passed.


In [23]:

# Define the final C10 column order.
# Group related fields together so the dataset is easy to inspect.

c10_columns = [
    # Canonical identifiers
    "franchise_id",
    "season",

    # Financial allocation and thresholds
    "total_apron_allocations",
    "first_apron_threshold",
    "second_apron_threshold",

    # Distance from each apron threshold
    "first_apron_space",
    "second_apron_space",

    # Financial position indicators
    "above_first_apron",
    "above_second_apron",

    # Hard-cap classifications
    "first_apron_hard_capped",
    "second_apron_hard_capped",

    # Historical applicability
    "second_apron_applicable",

    # Research provenance
    "verification_status",
    "source_url",
    "source_coverage"
]

# Build C10 using only the selected canonical fields.
c10 = c10_working[c10_columns].copy()

# Sort by season and franchise for consistent ordering.
c10 = (
    c10.sort_values(["season", "franchise_id"])
    .reset_index(drop=True)
)

print("C10 shape:", c10.shape)
print("\nC10 columns:")
print(c10.columns.tolist())

print("\nSample records:")
print(c10.head(5).to_string(index=False))


C10 shape: (450, 15)

C10 columns:
['franchise_id', 'season', 'total_apron_allocations', 'first_apron_threshold', 'second_apron_threshold', 'first_apron_space', 'second_apron_space', 'above_first_apron', 'above_second_apron', 'first_apron_hard_capped', 'second_apron_hard_capped', 'second_apron_applicable', 'verification_status', 'source_url', 'source_coverage']

Sample records:
 franchise_id    season  total_apron_allocations  first_apron_threshold  second_apron_threshold  first_apron_space  second_apron_space  above_first_apron above_second_apron first_apron_hard_capped second_apron_hard_capped  second_apron_applicable verification_status source_url source_coverage
   1610612737 2011-2012                 71683612               74307000                     NaN            2623388                 NaN              False                NaN                     NaN                      NaN                    False          unverified        NaN    not_verified
   1610612738 2011-2012        

In [24]:

# Every above-first-apron flag should agree with its corresponding space.
# Negative space means allocations exceed the threshold.
assert (
    c10["above_first_apron"]
    == (c10["first_apron_space"] < 0)
).all()

# The second-apron flag is only meaningful when the second apron applies.
modern = c10["second_apron_applicable"] == True

# For applicable seasons, confirm the flag agrees with the space.
assert (
    c10.loc[modern, "above_second_apron"]
    == (c10.loc[modern, "second_apron_space"] < 0)
).all()

# For historical seasons, the second-apron metrics must remain missing.
historical = ~modern

assert c10.loc[
    historical, "second_apron_threshold"
].isna().all()

assert c10.loc[
    historical, "second_apron_space"
].isna().all()

assert c10.loc[
    historical, "above_second_apron"
].isna().all()

print("C10 financial indicator validation passed.")


C10 financial indicator validation passed.


In [25]:

# Count missing observations in each canonical field.
# This helps distinguish expected historical gaps from data-quality problems.
missing_summary = (
    c10.isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values by column:")
print(missing_summary.to_string())

# Confirm the canonical keys are complete and unique.
assert c10[["franchise_id", "season"]].notna().all().all()

assert not c10.duplicated(
    ["franchise_id", "season"]
).any()

print("\nC10 key validation passed.")


Missing values by column:
second_apron_threshold      360
second_apron_space          360
above_second_apron          360
second_apron_hard_capped    360
first_apron_hard_capped     262
source_url                  262
franchise_id                  0
season                        0
total_apron_allocations       0
first_apron_threshold         0
first_apron_space             0
above_first_apron             0
second_apron_applicable       0
verification_status           0
source_coverage               0

C10 key validation passed.


In [26]:

# Define the output path for the new C10 canonical table.
# CANONICAL_DIR was established in Step 13.
c10_path = CANONICAL_DIR / "team_apron_seasons.csv"

# Save the 450-row, 15-column canonical dataset.
# index=False prevents pandas from writing the DataFrame index
# as an unnecessary additional CSV column.
c10.to_csv(c10_path, index=False)

print("Saved canonical table:", c10_path)
print("Shape:", c10.shape)


Saved canonical table: ../data/canonical/team_apron_seasons.csv
Shape: (450, 15)


In [27]:

# Reload the saved CSV to verify the persisted artifact.
c10_reloaded = pd.read_csv(c10_path)

# Check its dimensions.
assert c10_reloaded.shape == (450, 15)

# Confirm the canonical franchise-season key remains unique.
assert not c10_reloaded.duplicated(
    ["franchise_id", "season"]
).any()

# Confirm the financial fields survived the export.
assert c10_reloaded[
    ["total_apron_allocations", "first_apron_threshold", "first_apron_space"]
].notna().all().all()

# Check the expected historical missing values.
assert c10_reloaded["second_apron_threshold"].isna().sum() == 360
assert c10_reloaded["first_apron_hard_capped"].isna().sum() == 262

print("C10 export validation passed.")
print("Rows:", len(c10_reloaded))
print("Columns:", len(c10_reloaded.columns))
print("Seasons:", c10_reloaded["season"].nunique())


C10 export validation passed.
Rows: 450
Columns: 15
Seasons: 15


In [28]:

# Reload C10 directly from its saved canonical CSV.
# This ensures we're auditing the persisted dataset.
c10_audit = pd.read_csv(
    CANONICAL_DIR / "team_apron_seasons.csv"
)

# Select the common identifiers shared by C04 and C10.
key_cols = ["franchise_id", "season"]

# Verify each table has unique franchise-season observations.
assert not c10_audit.duplicated(key_cols).any()
assert not team_seasons.duplicated(key_cols).any()

# Match every C10 record against C04.
# validate="one_to_one" prevents duplicate keys on either side.
key_check = c10_audit[key_cols].merge(
    team_seasons[key_cols],
    on=key_cols,
    how="left",
    validate="one_to_one",
    indicator=True
)

print("C10 → C04 match results:")
print(key_check["_merge"].value_counts())

assert key_check["_merge"].eq("both").all()

print("\nStep 17A passed.")


C10 → C04 match results:
_merge
both          450
left_only       0
right_only      0
Name: count, dtype: int64

Step 17A passed.


In [29]:

# Count the number of franchise-season records in each dataset.
c04_coverage = (
    team_seasons.groupby("season")
    .size()
    .rename("c04_teams")
)

c10_coverage = (
    c10_audit.groupby("season")
    .size()
    .rename("c10_teams")
)

# Combine both coverage summaries.
# The outer join retains seasons that appear in either dataset.
season_coverage = pd.concat(
    [c04_coverage, c10_coverage],
    axis=1
).fillna(0).astype(int)

print(season_coverage.to_string())

# Confirm all 15 S19 seasons contain exactly 30 teams.
assert (c10_coverage == 30).all()

# Confirm no C10 season exists outside C04.
assert set(c10_coverage.index).issubset(
    set(c04_coverage.index)
)

print("\nStep 17B passed.")


           c04_teams  c10_teams
season                         
2000-2001         29          0
2001-2002         29          0
2002-2003         29          0
2003-2004         29          0
2004-2005         30          0
2005-2006         30          0
2006-2007         30          0
2007-2008         30          0
2008-2009         30          0
2009-2010         30          0
2010-2011         30          0
2011-2012         30         30
2012-2013         30         30
2013-2014         30         30
2014-2015         30         30
2015-2016         30         30
2016-2017         30         30
2017-2018         30         30
2018-2019         30         30
2019-2020         30         30
2020-2021         30         30
2021-2022         30         30
2022-2023         30         30
2023-2024         30         30
2024-2025         30         30
2025-2026         30         30

Step 17B passed.


In [30]:

# Select the financial fields from C04 that we want to compare.
c04_financial = team_seasons[
    [
        "franchise_id",
        "season",
        "team_payroll",
        "taxable_payroll",
        "salary_cap_hit",
        "s18_tax_threshold"
    ]
].copy()

# Join C10 to C04 using the validated canonical franchise-season key.
# We use an inner join because all 450 C10 observations have C04 matches.
financial_audit = c10_audit.merge(
    c04_financial,
    on=["franchise_id", "season"],
    how="inner",
    validate="one_to_one"
)

# Check that the merge preserved every C10 observation.
assert len(financial_audit) == 450

print("Financial audit shape:", financial_audit.shape)

print(financial_audit[
    [
        "season",
        "total_apron_allocations",
        "team_payroll",
        "taxable_payroll"
    ]
].head().to_string(index=False))


Financial audit shape: (450, 19)
   season  total_apron_allocations  team_payroll  taxable_payroll
2011-2012                 71683612      73669912       71683612.0
2011-2012                 76544614      79820530       76544614.0
2011-2012                 50463174      64875599       50463174.0
2011-2012                 58480742      65441008       58480742.0
2011-2012                 70787864      69548447       70787864.0


In [31]:

# Calculate the dollar difference between apron allocations and team payroll.
# Positive values mean apron allocations are higher than team payroll.
financial_audit["apron_minus_payroll"] = (
    financial_audit["total_apron_allocations"]
    - financial_audit["team_payroll"]
)

# Calculate the difference between apron allocations and taxable payroll.
# Missing taxable payroll values remain missing automatically.
financial_audit["apron_minus_taxable"] = (
    financial_audit["total_apron_allocations"]
    - financial_audit["taxable_payroll"]
)

# Calculate the relative difference as a percentage of team payroll.
# This makes comparisons across different NBA salary eras more meaningful.
financial_audit["apron_payroll_diff_pct"] = (
    financial_audit["apron_minus_payroll"]
    / financial_audit["team_payroll"]
    * 100
)

print(financial_audit[
    [
        "season",
        "franchise_id",
        "apron_minus_payroll",
        "apron_minus_taxable",
        "apron_payroll_diff_pct"
    ]
].head().to_string(index=False))


   season  franchise_id  apron_minus_payroll  apron_minus_taxable  apron_payroll_diff_pct
2011-2012    1610612737             -1986300                  0.0               -2.696216
2011-2012    1610612738             -3275916                  0.0               -4.104102
2011-2012    1610612739            -14412425                  0.0              -22.215479
2011-2012    1610612740             -6960266                  0.0              -10.635939
2011-2012    1610612741              1239417                  0.0                1.782092


In [32]:

# Aggregate the financial differences by season.
# Each named aggregation specifies:
# output_column = (source_column, aggregation_function)

financial_summary = (
    financial_audit.groupby("season")
    .agg(
        teams=("franchise_id", "nunique"),

        # Number of observations with available taxable payroll.
        taxable_available=("taxable_payroll", "count"),

        # Average signed dollar difference.
        avg_apron_minus_payroll=("apron_minus_payroll", "mean"),

        # Median absolute percentage difference.
        # abs() prevents positive and negative differences from cancelling.
        median_abs_diff_pct=(
            "apron_payroll_diff_pct",
            lambda s: s.abs().median()
        ),

        # Largest absolute percentage difference within the season.
        max_abs_diff_pct=(
            "apron_payroll_diff_pct",
            lambda s: s.abs().max()
        )
    )
    .reset_index()
)

print(financial_summary.round(2).to_string(index=False))


   season  teams  taxable_available  avg_apron_minus_payroll  median_abs_diff_pct  max_abs_diff_pct
2011-2012     30                 30              -4076939.40                 7.00             25.12
2012-2013     30                 30              -3594536.87                 2.82             24.21
2013-2014     30                 30              -2752889.20                 2.08             32.51
2014-2015     30                 30              -3686298.23                 3.67             16.71
2015-2016     30                 30                -58794.67                 1.45             19.43
2016-2017     30                 30               -848078.40                 1.19             16.58
2017-2018     30                 30              -1881753.00                 0.95             20.55
2018-2019     30                 30               -478294.70                 1.10             31.56
2019-2020     30                 30               7554312.33                 3.48             31.46


In [33]:

# Rank observations by the magnitude of the percentage difference.
# We use abs() because large negative differences matter too.
largest_discrepancies = (
    financial_audit.assign(
        abs_diff_pct=financial_audit[
            "apron_payroll_diff_pct"
        ].abs()
    )
    .sort_values("abs_diff_pct", ascending=False)
    .head(15)
)

print(largest_discrepancies[
    [
        "season",
        "franchise_id",
        "total_apron_allocations",
        "team_payroll",
        "apron_minus_payroll",
        "apron_payroll_diff_pct"
    ]
].round(2).to_string(index=False))


   season  franchise_id  total_apron_allocations  team_payroll  apron_minus_payroll  apron_payroll_diff_pct
2013-2014    1610612753                 53516731      79294567            -25777836                  -32.51
2018-2019    1610612737                104172575      79180081             24992494                   31.56
2019-2020    1610612763                129481312      98495848             30985464                   31.46
2019-2020    1610612766                123632035      96552033             27080002                   28.05
2019-2020    1610612765                131909920     104527576             27382344                   26.20
2011-2012    1610612758                 36652311      48950236            -12297925                  -25.12
2012-2013    1610612753                 66215396      87365418            -21150022                  -24.21
2011-2012    1610612764                 68146670      55386345             12760325                   23.04
2012-2013    1610612757     

In [34]:

# Count how many records have exactly matching financial values.
exact_matches = financial_audit["apron_minus_taxable"].eq(0).sum()

# Count nonmatching records.
nonmatches = financial_audit["apron_minus_taxable"].ne(0).sum()

# Find the largest absolute difference across all observations.
max_difference = financial_audit["apron_minus_taxable"].abs().max()

print("Exact matches:", exact_matches)
print("Nonmatches:", nonmatches)
print("Largest absolute difference:", max_difference)

# Show any mismatches so we can inspect them if necessary.
mismatches = financial_audit.loc[
    financial_audit["apron_minus_taxable"].ne(0),
    [
        "season",
        "franchise_id",
        "total_apron_allocations",
        "taxable_payroll",
        "apron_minus_taxable"
    ]
]

print("\nMismatching observations:")
print(mismatches.to_string(index=False))


Exact matches: 204
Nonmatches: 246
Largest absolute difference: 33705369.0

Mismatching observations:
   season  franchise_id  total_apron_allocations  taxable_payroll  apron_minus_taxable
2015-2016    1610612739                107028394      107028354.0                 40.0
2015-2016    1610612740                 83075760       81425760.0            1650000.0
2015-2016    1610612744                 95777682       95879899.0            -102217.0
2015-2016    1610612748                 84619637       84629571.0              -9934.0
2015-2016    1610612750                 70902855       70652855.0             250000.0
2016-2017    1610612737                 95563799       95576245.0             -12446.0
2016-2017    1610612739                126590163      126590785.0               -622.0
2016-2017    1610612740                103067816      101485185.0            1582631.0
2016-2017    1610612741                 93175538       93281333.0            -105795.0
2016-2017    1610612742     

In [35]:

# Filter to the season with the largest positive average difference.
season_2020 = financial_audit.loc[
    financial_audit["season"] == "2019-2020"
].copy()

# Count the direction of the differences.
print("Positive differences:", season_2020["apron_minus_payroll"].gt(0).sum())
print("Negative differences:", season_2020["apron_minus_payroll"].lt(0).sum())
print("Exact matches:", season_2020["apron_minus_payroll"].eq(0).sum())

# Summarize the distribution rather than relying only on the average.
print("\n2019-20 dollar differences:")
print(
    season_2020["apron_minus_payroll"]
    .describe()
    .round(2)
    .to_string()
)


Positive differences: 28
Negative differences: 2
Exact matches: 0

2019-20 dollar differences:
count          30.00
mean      7554312.33
std       8901638.79
min      -1351409.00
25%       2294777.75
50%       4270271.50
75%       9883640.50
max      30985464.00


In [36]:

# Group C10 by season to inspect the threshold values.
threshold_summary = (
    c10_audit.groupby("season")
    .agg(
        teams=("franchise_id", "nunique"),
        first_min=("first_apron_threshold", "min"),
        first_max=("first_apron_threshold", "max"),
        second_min=("second_apron_threshold", "min"),
        second_max=("second_apron_threshold", "max")
    )
    .reset_index()
)

# Calculate the difference between the highest and lowest
# reported threshold within each season.
threshold_summary["first_spread"] = (
    threshold_summary["first_max"]
    - threshold_summary["first_min"]
)

threshold_summary["second_spread"] = (
    threshold_summary["second_max"]
    - threshold_summary["second_min"]
)

# Verify all teams share the same first-apron threshold per season.
assert threshold_summary["first_spread"].eq(0).all()

# For modern seasons, verify the second-apron threshold is consistent.
modern_thresholds = threshold_summary[
    threshold_summary["season"] >= "2023-2024"
]

assert modern_thresholds["second_spread"].eq(0).all()

print(threshold_summary.to_string(index=False))
print("\nStep 19 internal threshold consistency passed.")


   season  teams  first_min  first_max  second_min  second_max  first_spread  second_spread
2011-2012     30   74307000   74307000         NaN         NaN             0            NaN
2012-2013     30   74307000   74307000         NaN         NaN             0            NaN
2013-2014     30   75748000   75748000         NaN         NaN             0            NaN
2014-2015     30   80829000   80829000         NaN         NaN             0            NaN
2015-2016     30   88740000   88740000         NaN         NaN             0            NaN
2016-2017     30  117287000  117287000         NaN         NaN             0            NaN
2017-2018     30  125266000  125266000         NaN         NaN             0            NaN
2018-2019     30  129817000  129817000         NaN         NaN             0            NaN
2019-2020     30  138928000  138928000         NaN         NaN             0            NaN
2020-2021     30  138928000  138928000         NaN         NaN             0    

In [37]:

# Summarize how many observations belong to each research status.
verification_summary = (
    c10_audit.groupby("verification_status")
    .agg(
        teams=("franchise_id", "count"),
        first_known=("first_apron_hard_capped", "count"),
        second_known=("second_apron_hard_capped", "count"),
        source_urls=("source_url", "count")
    )
    .reset_index()
)

print(verification_summary.to_string(index=False))


        verification_status  teams  first_known  second_known  source_urls
  confirmed_source_snapshot    171          171            73          171
source_reported_no_hard_cap     17           17            17           17
                 unverified    262            0             0            0


In [38]:

# Define the historical and modern NBA apron periods.
historical = c10_audit["season"] < "2023-2024"
modern = ~historical

# Historical second-apron classifications must be inapplicable.
assert c10_audit.loc[
    historical, "second_apron_hard_capped"
].isna().all()

assert (
    c10_audit.loc[historical, "second_apron_applicable"] == False
).all()

# Modern seasons should have source-reported classifications.
assert c10_audit.loc[
    modern, "first_apron_hard_capped"
].notna().all()

assert c10_audit.loc[
    modern, "second_apron_hard_capped"
].notna().all()

print("Historical applicability validation passed.")


Historical applicability validation passed.


In [39]:

# Create a comparison table for the modern apron era.
modern_hard_caps = c10_audit.loc[modern].copy()

# Count each combination of financial position and hard-cap status.
# dropna=False ensures missing classifications would remain visible.
hard_cap_comparison = (
    modern_hard_caps.groupby(
        ["above_first_apron", "first_apron_hard_capped"],
        dropna=False
    )
    .size()
    .rename("teams")
    .reset_index()
)

print("First-apron position vs. hard-cap classification:")
print(hard_cap_comparison.to_string(index=False))

print("\nSecond-apron position vs. hard-cap classification:")

second_comparison = (
    modern_hard_caps.groupby(
        ["above_second_apron", "second_apron_hard_capped"],
        dropna=False
    )
    .size()
    .rename("teams")
    .reset_index()
)

print(second_comparison.to_string(index=False))


First-apron position vs. hard-cap classification:
 above_first_apron  first_apron_hard_capped  teams
             False                    False     16
             False                     True     57
              True                    False     17

Second-apron position vs. hard-cap classification:
 above_second_apron  second_apron_hard_capped  teams
              False                     False     65
              False                      True     16
               True                     False      9


In [40]:

# Start with C04 as our team-season analytical foundation.
# A left join preserves all historical C04 observations.
readiness_df = team_seasons.merge(
    c10_audit,
    on=["franchise_id", "season"],
    how="left",
    validate="one_to_one"
)

# Verify that no team-season records were lost or duplicated.
assert len(readiness_df) == len(team_seasons)

# Define the variables we expect to use in Phase B.
feature_columns = [
    "wins",
    "win_pct",
    "made_playoffs",
    "team_payroll",
    "taxable_payroll",
    "total_apron_allocations",
    "first_apron_threshold",
    "first_apron_space",
    "above_first_apron",
    "second_apron_threshold",
    "above_second_apron",
    "first_apron_hard_capped",
    "second_apron_hard_capped"
]

# Count nonmissing observations and calculate coverage percentages.
feature_readiness = pd.DataFrame({
    "available": readiness_df[feature_columns].notna().sum(),
    "total": len(readiness_df)
})

feature_readiness["coverage_pct"] = (
    feature_readiness["available"]
    / feature_readiness["total"]
    * 100
).round(1)

print(feature_readiness.to_string())


                          available  total  coverage_pct
wins                            776    776         100.0
win_pct                         776    776         100.0
made_playoffs                   776    776         100.0
team_payroll                    776    776         100.0
taxable_payroll                 450    776          58.0
total_apron_allocations         450    776          58.0
first_apron_threshold           450    776          58.0
first_apron_space               450    776          58.0
above_first_apron               450    776          58.0
second_apron_threshold           90    776          11.6
above_second_apron               90    776          11.6
first_apron_hard_capped         188    776          24.2
second_apron_hard_capped         90    776          11.6


In [41]:

# Additional C04 financial variables needed for our research.
financial_features = [
    "salary_cap_hit",
    "salary_cap_space",
    "s18_tax_threshold",
    "taxable_payroll",
    "tax_space",
    "tax_paid",
    "estimated_tax_bill",
    "total_estimated_cost",
    "active_salary",
    "dead_salary"
]

# Calculate the number of available observations.
financial_readiness = pd.DataFrame({
    "available": readiness_df[financial_features].notna().sum(),
    "total": len(readiness_df)
})

# Calculate historical coverage as a percentage.
financial_readiness["coverage_pct"] = (
    financial_readiness["available"]
    / financial_readiness["total"]
    * 100
).round(1)

print(financial_readiness.to_string())


                      available  total  coverage_pct
salary_cap_hit              776    776         100.0
salary_cap_space             90    776          11.6
s18_tax_threshold           688    776          88.7
taxable_payroll             450    776          58.0
tax_space                   450    776          58.0
tax_paid                    268    776          34.5
estimated_tax_bill           94    776          12.1
total_estimated_cost        450    776          58.0
active_salary               450    776          58.0
dead_salary                 416    776          53.6


In [42]:

# Identify observations with reported tax payments.
has_reported_tax = readiness_df["tax_paid"].notna()

# Identify observations with estimated tax bills.
has_estimated_tax = readiness_df["estimated_tax_bill"].notna()

# Count the overlap and unique coverage.
both = (has_reported_tax & has_estimated_tax).sum()
reported_only = (has_reported_tax & ~has_estimated_tax).sum()
estimated_only = (~has_reported_tax & has_estimated_tax).sum()
neither = (~has_reported_tax & ~has_estimated_tax).sum()

print("Reported tax only:", reported_only)
print("Estimated tax only:", estimated_only)
print("Both available:", both)
print("Neither available:", neither)

# Count team-seasons with at least one tax-cost measure.
total_tax_coverage = (has_reported_tax | has_estimated_tax).sum()

print("\nTotal tax-cost coverage:", total_tax_coverage)
print(
    "Coverage percentage:",
    round(total_tax_coverage / len(readiness_df) * 100, 1)
)

# Ensure the categories account for every observation.
assert reported_only + estimated_only + both + neither == len(readiness_df)


Reported tax only: 268
Estimated tax only: 94
Both available: 0
Neither available: 414

Total tax-cost coverage: 362
Coverage percentage: 46.6


In [43]:

# Create a temporary indicator for whether either tax-cost
# measure is available for a team-season.
tax_coverage = readiness_df.assign(
    has_tax_cost=(
        readiness_df["tax_paid"].notna()
        | readiness_df["estimated_tax_bill"].notna()
    )
)

# Summarize coverage across seasons.
tax_season_summary = (
    tax_coverage.groupby("season")
    .agg(
        teams=("franchise_id", "count"),
        reported=("tax_paid", "count"),
        estimated=("estimated_tax_bill", "count"),
        either=("has_tax_cost", "sum")
    )
    .reset_index()
)

print(tax_season_summary.to_string(index=False))


   season  teams  reported  estimated  either
2000-2001     29         0          0       0
2001-2002     29         0          0       0
2002-2003     29        29          0      29
2003-2004     29        29          0      29
2004-2005     30        30          0      30
2005-2006     30        30          0      30
2006-2007     30        30          0      30
2007-2008     30        30          0      30
2008-2009     30        30          0      30
2009-2010     30        30          0      30
2010-2011     30        30          0      30
2011-2012     30         0          7       7
2012-2013     30         0          6       6
2013-2014     30         0          6       6
2014-2015     30         0          5       5
2015-2016     30         0          7       7
2016-2017     30         0          2       2
2017-2018     30         0          4       4
2018-2019     30         0          5       5
2019-2020     30         0          4       4
2020-2021     30         0        